# HW1 – 2025 – Simulation | Phase 1: Simple Criteria
## Domain: Video Games (PC & PlayStation)

**Author:** Georgios Kitsakis  
**Institution:** Athens University of Economics and Business (AUEB)

---

**Phase 1** uses only the most obvious filter per segment to generate ratings.
The goal is to see if LDA can already find the 5 segments with simple, broad signals.

| # | Segment | Simple like condition |
|---|---|---|
| 1 | **PC Purist** | game is on PC or BOTH |
| 2 | **PlayStation Fan** | game is on PS or BOTH |
| 3 | **Cross-Platform Gamer** | game is on BOTH |
| 4 | **Budget Gamer** | price ≤ €20 |
| 5 | **Casual / Family Gamer** | PEGI age rating '3' or '7' |

→ See **Phase 2** notebook for harder criteria.

In [1]:
!pip install tomotopy -q

In [2]:
import random
import csv
import numpy as np
import pandas as pd
import tomotopy as tp
from dataclasses import dataclass
from typing import List, Tuple
import warnings
warnings.filterwarnings('ignore')

SEG_NAMES = {
    1: 'PC Purist',
    2: 'PlayStation Fan',
    3: 'Cross-Platform Gamer',
    4: 'Budget Gamer',
    5: 'Casual / Family Gamer'
}

---
## 1. `generate_entities()`

Each game has **10 distinct attributes**: title, platform, genre, price_eur, metacritic,
avg_playtime_h, is_multiplayer, is_exclusive, age_rating, release_year.

In [3]:
@dataclass
class VideoGame:
    title: str
    platform: str
    genre: str
    price_eur: float
    metacritic: int
    avg_playtime_h: float
    is_multiplayer: bool
    is_exclusive: bool
    age_rating: str
    release_year: int

In [ ]:
def generate_entities(
    game_num: int = 200,
    genre_options: List[str] = ['Action', 'RPG', 'Sports', 'Strategy', 'Horror',
                                 'Adventure', 'Simulation', 'Fighting', 'Puzzle', 'Racing'],
    platform_options: List[str] = ['PC', 'PS', 'BOTH'],
    platform_distro: List[float] = [0.35, 0.30, 0.35],
    price_gaussian_params: Tuple[float, float] = (35, 18),
    metacritic_gaussian_params: Tuple[float, float] = (68, 15),
    playtime_gaussian_params: Tuple[float, float] = (25, 20),
    multiplayer_prob: float = 0.45,
    age_ratings: List[str] = ['3', '7', '12', '16', '18'],
    year_range: Tuple[int, int] = (2010, 2024)
) -> List[VideoGame]:
    """
    Generates a list of synthetic video game entities.

    Each game has 10 distinct attributes:
        title            unique string identifier (e.g. 'Game_0001')
        platform         'PC', 'PS', or 'BOTH' (sampled from platform_distro)
        genre            primary genre from genre_options
        price_eur        retail price in EUR (Gaussian, clipped to [5, 80])
        metacritic       Metacritic score 0-100 (Gaussian, clipped)
        avg_playtime_h   average playtime in hours (Gaussian, clipped to [1, 200])
        is_multiplayer   True if the game supports online multiplayer
        is_exclusive     True if platform-exclusive (platform != 'BOTH')
        age_rating       PEGI rating: '3', '7', '12', '16', or '18'
        release_year     release year, sampled uniformly in year_range

    Args:
        game_num (int):                    Number of games to generate. Default: 200.
        genre_options (List[str]):         Available genres.
        platform_options (List[str]):      Platform choices.
        platform_distro (List[float]):     Probability distribution over platforms.
        price_gaussian_params (Tuple):     (mean, std) for price in EUR.
        metacritic_gaussian_params (Tuple):(mean, std) for Metacritic score.
        playtime_gaussian_params (Tuple):  (mean, std) for avg playtime in hours.
        multiplayer_prob (float):          Probability of multiplayer support.
        age_ratings (List[str]):           PEGI rating options.
        year_range (Tuple[int, int]):      (min_year, max_year) for release year.

    Returns:
        List[VideoGame]: A list of generated VideoGame objects.
    """
    games = []
    for i in range(game_num):
        platform_idx = np.random.choice(len(platform_options), p=platform_distro)
        platform     = platform_options[platform_idx]
        genre        = random.choice(genre_options)
        price        = round(float(np.clip(random.gauss(*price_gaussian_params), 5.0, 80.0)), 2)
        metacritic   = int(np.clip(random.gauss(*metacritic_gaussian_params), 0, 100))
        playtime     = round(float(np.clip(random.gauss(*playtime_gaussian_params), 1.0, 200.0)), 1)
        is_multi     = random.random() < multiplayer_prob
        is_excl      = platform != 'BOTH'
        age_rating   = random.choice(age_ratings)
        year         = random.randint(*year_range)
        games.append(VideoGame(
            title=f'Game_{i+1:04d}', platform=platform, genre=genre,
            price_eur=price, metacritic=metacritic, avg_playtime_h=playtime,
            is_multiplayer=is_multi, is_exclusive=is_excl,
            age_rating=age_rating, release_year=year
        ))
    return games

In [5]:
games = generate_entities(game_num=200)
print(f'Generated {len(games)} games.')
print(vars(games[0]))
print('Platform distribution:', pd.Series([g.platform for g in games]).value_counts().to_dict())

Generated 200 games.
{'title': 'Game_0001', 'platform': 'BOTH', 'genre': 'Racing', 'price_eur': 53.66, 'metacritic': 75, 'avg_playtime_h': 30.6, 'is_multiplayer': True, 'is_exclusive': False, 'age_rating': '12', 'release_year': 2015}
Platform distribution: {'BOTH': 76, 'PC': 64, 'PS': 60}


---
## 2. `generate_users()`

One helper function per segment, all called from `generate_users()`.

In [6]:
@dataclass
class User:
    segment: int
    age: int
    gender: str

In [7]:
def generate_users_segment1(user_num: int = 200) -> List[User]:
    """
    Generates users for Segment 1: PC Purist.

    Characteristics:
    - Dedicated PC gamers who only care about games on PC or BOTH platforms
    - Will like any game that runs on PC (simple mode)
    - 50% Male, 50% Female
    - Age: Gaussian(30, 6)

    Args:
        user_num (int): Number of users. Default: 200.
    Returns:
        List[User]: Segment 1 users.
    """
    return [User(segment=1, age=max(10, int(random.gauss(30, 6))),
                 gender=random.choice(['M','F'])) for _ in range(user_num)]


def generate_users_segment2(user_num: int = 200) -> List[User]:
    """
    Generates users for Segment 2: PlayStation Fan.

    Characteristics:
    - Console loyalists who only buy games for their PlayStation (PS or BOTH)
    - Will like any game that runs on PlayStation (simple mode)
    - 50% Male, 50% Female
    - Age: Gaussian(25, 7)

    Args:
        user_num (int): Number of users. Default: 200.
    Returns:
        List[User]: Segment 2 users.
    """
    return [User(segment=2, age=max(10, int(random.gauss(25, 7))),
                 gender=random.choice(['M','F'])) for _ in range(user_num)]


def generate_users_segment3(user_num: int = 200) -> List[User]:
    """
    Generates users for Segment 3: Cross-Platform Gamer.

    Characteristics:
    - Own both PC and PlayStation; only buy games on BOTH platforms
    - Will like any game available on BOTH platforms (simple mode)
    - 50% Male, 50% Female
    - Age: Gaussian(22, 5)

    Args:
        user_num (int): Number of users. Default: 200.
    Returns:
        List[User]: Segment 3 users.
    """
    return [User(segment=3, age=max(10, int(random.gauss(22, 5))),
                 gender=random.choice(['M','F'])) for _ in range(user_num)]


def generate_users_segment4(user_num: int = 200) -> List[User]:
    """
    Generates users for Segment 4: Budget Gamer.

    Characteristics:
    - Price-first players: any platform, any genre, as long as it is cheap (<=20 EUR)
    - Will like any game priced at or below 20 EUR (simple mode)
    - 50% Male, 50% Female
    - Age: Gaussian(20, 8)

    Args:
        user_num (int): Number of users. Default: 200.
    Returns:
        List[User]: Segment 4 users.
    """
    return [User(segment=4, age=max(10, int(random.gauss(20, 8))),
                 gender=random.choice(['M','F'])) for _ in range(user_num)]


def generate_users_segment5(user_num: int = 200) -> List[User]:
    """
    Generates users for Segment 5: Casual / Family Gamer.

    Characteristics:
    - Occasional players who want fun, short, family-friendly games
    - Will like any game with PEGI age rating '3' or '7' (simple mode)
    - Any platform is fine
    - 50% Male, 50% Female
    - Age: Gaussian(38, 10)

    Args:
        user_num (int): Number of users. Default: 200.
    Returns:
        List[User]: Segment 5 users.
    """
    return [User(segment=5, age=max(10, int(random.gauss(38, 10))),
                 gender=random.choice(['M','F'])) for _ in range(user_num)]

In [ ]:
def generate_users(user_num: int = 1000) -> List[User]:
    """
    Generates a population of users spread across 5 distinct customer segments.
    Users are distributed equally (user_num // 5 per segment).

    Internally calls:
        generate_users_segment1()  PC Purist
        generate_users_segment2()  PlayStation Fan
        generate_users_segment3()  Cross-Platform Gamer
        generate_users_segment4()  Budget Gamer
        generate_users_segment5()  Casual / Family Gamer

    Args:
        user_num (int): Total number of users to generate. Default: 1000.

    Returns:
        List[User]: A shuffled list of User objects tagged with segment id (1–5).
    """
    n = user_num // 5
    users = (
        generate_users_segment1(n) +
        generate_users_segment2(n) +
        generate_users_segment3(n) +
        generate_users_segment4(n) +
        generate_users_segment5(n)
    )
    random.shuffle(users)
    return users

In [9]:
users = generate_users(user_num=1000)
counts = pd.Series([u.segment for u in users]).value_counts().sort_index()
for seg, cnt in counts.items():
    print(f'  Segment {seg} – {SEG_NAMES[seg]:25s}: {cnt} users')

  Segment 1 – PC Purist                : 200 users
  Segment 2 – PlayStation Fan          : 200 users
  Segment 3 – Cross-Platform Gamer     : 200 users
  Segment 4 – Budget Gamer             : 200 users
  Segment 5 – Casual / Family Gamer    : 200 users


---
## 3. `generate_ratings()` — Simple Criteria

Each segment has its own helper. The like condition is the **simplest possible**:
only platform / price / age_rating is checked. Any game passing that check is liked.
After the ground-truth is decided, the rating is flipped with probability `noise`.

In [10]:
def _make_row(user, game, rating, reason):
    return {'segment': user.segment, 'age': user.age, 'gender': user.gender,
            'game': game.title, 'platform': game.platform, 'genre': game.genre,
            'price_eur': game.price_eur, 'metacritic': game.metacritic,
            'avg_playtime_h': game.avg_playtime_h, 'is_multiplayer': game.is_multiplayer,
            'is_exclusive': game.is_exclusive, 'age_rating': game.age_rating,
            'release_year': game.release_year, 'rating': rating, 'reason': reason}


def generate_ratings_segment1(users, games, pairs, noise):
    """
    Generates ratings for Segment 1: PC Purist — Simple Criteria.

    Criteria:
    - Like if game is on PC or BOTH platform
    - Dislike otherwise
    - noise% chance of rating flip

    Args:
        users: full user list
        games: full game list
        pairs: (user_idx, game_idx) tuples for this segment
        noise: flip probability
    Returns:
        list of row dicts
    """
    rows = []
    for u_idx, g_idx in pairs:
        user, game = users[u_idx], games[g_idx]
        if game.platform in ('PC', 'BOTH'):
            rating, reason = 1, 'PC platform'
        else:
            rating, reason = -1, 'Not on PC'
        if random.random() < noise:
            rating *= -1; reason += ' [NOISE FLIP]'
        rows.append(_make_row(user, game, rating, reason))
    return rows


def generate_ratings_segment2(users, games, pairs, noise):
    """
    Generates ratings for Segment 2: PlayStation Fan — Simple Criteria.

    Criteria:
    - Like if game is on PS or BOTH platform
    - Dislike otherwise
    - noise% chance of rating flip

    Args:
        users: full user list
        games: full game list
        pairs: (user_idx, game_idx) tuples for this segment
        noise: flip probability
    Returns:
        list of row dicts
    """
    rows = []
    for u_idx, g_idx in pairs:
        user, game = users[u_idx], games[g_idx]
        if game.platform in ('PS', 'BOTH'):
            rating, reason = 1, 'PS platform'
        else:
            rating, reason = -1, 'Not on PlayStation'
        if random.random() < noise:
            rating *= -1; reason += ' [NOISE FLIP]'
        rows.append(_make_row(user, game, rating, reason))
    return rows


def generate_ratings_segment3(users, games, pairs, noise):
    """
    Generates ratings for Segment 3: Cross-Platform Gamer — Simple Criteria.

    Criteria:
    - Like if game is available on BOTH platforms
    - Dislike otherwise
    - noise% chance of rating flip

    Args:
        users: full user list
        games: full game list
        pairs: (user_idx, game_idx) tuples for this segment
        noise: flip probability
    Returns:
        list of row dicts
    """
    rows = []
    for u_idx, g_idx in pairs:
        user, game = users[u_idx], games[g_idx]
        if game.platform == 'BOTH':
            rating, reason = 1, 'BOTH platform'
        else:
            rating, reason = -1, 'Not on both platforms'
        if random.random() < noise:
            rating *= -1; reason += ' [NOISE FLIP]'
        rows.append(_make_row(user, game, rating, reason))
    return rows


def generate_ratings_segment4(users, games, pairs, noise):
    """
    Generates ratings for Segment 4: Budget Gamer — Simple Criteria.

    Criteria:
    - Like if price_eur <= 20
    - Dislike otherwise
    - noise% chance of rating flip

    Args:
        users: full user list
        games: full game list
        pairs: (user_idx, game_idx) tuples for this segment
        noise: flip probability
    Returns:
        list of row dicts
    """
    rows = []
    for u_idx, g_idx in pairs:
        user, game = users[u_idx], games[g_idx]
        if game.price_eur <= 20:
            rating, reason = 1, 'Cheap enough'
        else:
            rating, reason = -1, 'Too expensive'
        if random.random() < noise:
            rating *= -1; reason += ' [NOISE FLIP]'
        rows.append(_make_row(user, game, rating, reason))
    return rows


def generate_ratings_segment5(users, games, pairs, noise):
    """
    Generates ratings for Segment 5: Casual / Family Gamer — Simple Criteria.

    Criteria:
    - Like if age_rating is '3' or '7'
    - Dislike otherwise
    - noise% chance of rating flip

    Args:
        users: full user list
        games: full game list
        pairs: (user_idx, game_idx) tuples for this segment
        noise: flip probability
    Returns:
        list of row dicts
    """
    rows = []
    for u_idx, g_idx in pairs:
        user, game = users[u_idx], games[g_idx]
        if game.age_rating in ('3', '7'):
            rating, reason = 1, 'Family age rating'
        else:
            rating, reason = -1, 'Age rating too high for family'
        if random.random() < noise:
            rating *= -1; reason += ' [NOISE FLIP]'
        rows.append(_make_row(user, game, rating, reason))
    return rows

In [ ]:
def generate_ratings(
    users: List[User],
    games: List[VideoGame],
    n_ratings: int = 10000,
    noise: float = 0.05,
    output_file: str = 'ratings_simple.csv'
) -> None:
    """
    Generates binary like (+1) / dislike (-1) ratings using SIMPLE criteria
    and writes them to a CSV file.

    Simple like condition per segment:
        Segment 1  PC Purist         : game.platform in ('PC', 'BOTH')
        Segment 2  PlayStation Fan    : game.platform in ('PS', 'BOTH')
        Segment 3  Cross-Platform     : game.platform == 'BOTH'
        Segment 4  Budget Gamer       : game.price_eur <= 20
        Segment 5  Casual/Family      : game.age_rating in ('3', '7')

    After the ground-truth is decided, the rating is flipped with probability
    `noise`, simulating real-world inconsistent user behavior.

    Internally calls:
        generate_ratings_segment1() through generate_ratings_segment5()

    Args:
        users (List[User]):       User objects.
        games (List[VideoGame]):  VideoGame objects.
        n_ratings (int):          Total (user, game) pairs to rate. Default: 10000.
        noise (float):            Probability [0,1] of flipping a rating. Default: 0.05.
        output_file (str):        CSV filename. Default: 'ratings_simple.csv'.

    Returns:
        None — writes results to output_file.
    """
    all_pairs = [(u, g) for u in range(len(users)) for g in range(len(games))]
    sampled   = random.sample(all_pairs, min(n_ratings, len(all_pairs)))

    seg_pairs = {s: [] for s in range(1, 6)}
    for u_idx, g_idx in sampled:
        seg_pairs[users[u_idx].segment].append((u_idx, g_idx))

    handlers = {
        1: generate_ratings_segment1,
        2: generate_ratings_segment2,
        3: generate_ratings_segment3,
        4: generate_ratings_segment4,
        5: generate_ratings_segment5,
    }

    all_rows = []
    for s in range(1, 6):
        all_rows.extend(handlers[s](users, games, seg_pairs[s], noise))
    random.shuffle(all_rows)

    fieldnames = ['segment','age','gender','game','platform','genre','price_eur',
                  'metacritic','avg_playtime_h','is_multiplayer','is_exclusive',
                  'age_rating','release_year','rating','reason']
    with open(output_file, 'w', newline='', encoding='utf-8') as fw:
        writer = csv.DictWriter(fw, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(all_rows)

    total    = len(all_rows)
    positive = sum(1 for r in all_rows if r['rating'] == 1)
    print(f'Total ratings    : {total:,}')
    print(f'Positive (like)  : {positive:,}  ({positive/total:.1%})')
    print(f'Negative (dislike): {total-positive:,}  ({(total-positive)/total:.1%})')
    print(f'Noise level      : {noise:.0%}')
    print(f'Saved to         : {output_file}')

In [12]:
generate_ratings(users, games, n_ratings=10000, noise=0.05, output_file='ratings_simple.csv')

df = pd.read_csv('ratings_simple.csv')
print()
print('Like rate per segment:')
for seg, grp in df.groupby('segment'):
    print(f'  Segment {seg} – {SEG_NAMES[seg]:25s}: {(grp["rating"]==1).mean():.1%} likes  ({len(grp):,} ratings)')

Total ratings    : 10,000
Positive (like)  : 4,682  (46.8%)
Negative (dislike): 5,318  (53.2%)
Noise level      : 5%
Saved to         : ratings_simple.csv

Like rate per segment:
  Segment 1 – PC Purist                : 68.0% likes  (1,996 ratings)
  Segment 2 – PlayStation Fan          : 65.8% likes  (2,012 ratings)
  Segment 3 – Cross-Platform Gamer     : 38.5% likes  (1,997 ratings)
  Segment 4 – Budget Gamer             : 24.6% likes  (1,997 ratings)
  Segment 5 – Casual / Family Gamer    : 37.0% likes  (1,998 ratings)


In [13]:
df.head(10)

,segment,age,gender,game,platform,genre,price_eur,metacritic,avg_playtime_h,is_multiplayer,is_exclusive,age_rating,release_year,rating,reason
0,4,23,F,Game_0116,PS,Fighting,18.37,46,14.4,False,True,7,2017,1,Cheap enough
1,1,23,F,Game_0030,PC,Racing,45.98,99,12.6,True,True,18,2010,1,PC platform
2,2,35,F,Game_0062,PC,Action,61.41,59,32.7,True,True,18,2016,-1,Not on PlayStation
3,1,26,M,Game_0078,BOTH,RPG,43.28,77,38.4,True,False,3,2015,1,PC platform
4,3,24,F,Game_0115,PC,Simulation,5.52,24,57.3,True,True,18,2013,-1,Not on both platforms
5,5,35,F,Game_0035,BOTH,Puzzle,26.59,73,22.0,False,False,7,2010,1,Family age rating
6,5,53,M,Game_0013,BOTH,Puzzle,49.11,55,12.7,True,False,3,2019,-1,Family age rating [NOISE FLIP]
7,2,20,F,Game_0038,BOTH,Sports,13.20,76,50.0,False,False,3,2024,1,PS platform
8,3,30,M,Game_0052,BOTH,Strategy,24.86,30,34.4,False,False,3,2020,1,BOTH platform
9,5,32,F,Game_0116,PS,Fighting,18.37,46,14.4,False,True,7,2017,1,Family age rating


---
## 4. `learn_segments()` — LDA with Anchor Words

Each user → **document**. Each rated game → **token**: `Game_0042_PC_Strategy_LIKE`

**Anchor words** guide LDA toward the known segments:
- `_PC_` tokens → PC Purist
- `_PS_` tokens → PlayStation Fan  
- `_BOTH_` tokens → Cross-Platform Gamer
- Cheap-game LIKE tokens → Budget Gamer
- Family-rated LIKE tokens → Casual/Family Gamer

**Expected result with simple criteria:** Platform segments (1,2,3) should separate very
cleanly. Budget and Casual may blur since price and age_rating don't map to a unique platform.

In [14]:
def learn_segments(
    ratings_file: str = 'ratings_simple.csv',
    games: List[VideoGame] = None,
    k: int = 5,
    n_iter: int = 500,
    top_n_words: int = 10
) -> None:
    """
    Reads the ratings CSV and uses LDA (tomotopy) to discover the underlying
    customer segments from the like/dislike patterns.

    Method
    ------
    1. Each user → one DOCUMENT.
       Each rated game → TOKEN: "<title>_<platform>_<genre>_LIKE/DISLIKE"
    2. LDA is trained with k topics.
    3. ANCHOR WORDS seed each topic toward a known segment:
       - Topic 0: _PC_ tokens   → PC Purist
       - Topic 1: _PS_ tokens   → PlayStation Fan
       - Topic 2: _BOTH_ tokens → Cross-Platform Gamer
       - Topic 3: cheap-game LIKE tokens → Budget Gamer
       - Topic 4: family-rated LIKE tokens → Casual/Family Gamer
    4. Top-N words per topic are printed with a summary.
    5. Cross-tab of true segment vs discovered topic is printed.

    Args:
        ratings_file (str):      Path to the CSV from generate_ratings().
        games (List[VideoGame]): Game objects (for anchor lookup).
        k (int):                 Number of LDA topics. Default: 5.
        n_iter (int):            Training iterations. Default: 500.
        top_n_words (int):       Top words per topic. Default: 10.

    Returns:
        None – prints topic summaries and cross-tab.
    """
    df = pd.read_csv(ratings_file)
    game_lookup = {g.title: g for g in games} if games else {}

    # Build user documents
    df['_uid'] = df['segment'].astype(str) + '_' + df['age'].astype(str) + '_' + df['gender']
    user_docs = {}
    for uid, grp in df.groupby('_uid'):
        tokens = [f"{r['game']}_{r['platform']}_{r['genre']}_{'LIKE' if r['rating']==1 else 'DISLIKE'}"
                  for _, r in grp.iterrows()]
        if tokens:
            user_docs[uid] = tokens

    print(f'Documents (unique users) : {len(user_docs)}')
    print(f'Avg tokens per document  : {np.mean([len(v) for v in user_docs.values()]):.1f}')

    # Anchor words
    all_tokens = set(t for doc in user_docs.values() for t in doc)
    anchor_pc   = [t for t in all_tokens if '_PC_' in t][:20]
    anchor_ps   = [t for t in all_tokens if '_PS_' in t][:20]
    anchor_both = [t for t in all_tokens if '_BOTH_' in t][:20]
    anchor_budget, anchor_casual = [], []
    if game_lookup:
        for t in all_tokens:
            parts = t.split('_')
            g_title = parts[0] + '_' + parts[1]
            if g_title not in game_lookup: continue
            g = game_lookup[g_title]
            if g.price_eur <= 20 and t.endswith('LIKE'):   anchor_budget.append(t)
            if g.age_rating in ('3','7') and t.endswith('LIKE'): anchor_casual.append(t)
    anchor_budget = anchor_budget[:20]
    anchor_casual = anchor_casual[:20]

    print(f'\nAnchor sizes: PC={len(anchor_pc)}, PS={len(anchor_ps)}, '
          f'BOTH={len(anchor_both)}, Budget={len(anchor_budget)}, Casual={len(anchor_casual)}')

    # Train LDA
    lda = tp.LDAModel(k=k, seed=42)
    for doc_tokens in user_docs.values():
        lda.add_doc(doc_tokens)
    print('\nTraining LDA...')
    for i in range(0, n_iter, 50):
        lda.train(50)
        print(f'  Iteration {i+50:4d}  |  log-likelihood: {lda.ll_per_word:.4f}')

    # Print topics
    print('\n' + '='*65)
    print('  LDA DISCOVERED TOPICS')
    print('='*65)
    for tid in range(lda.k):
        top_words  = [p[0] for p in lda.get_topic_words(tid, top_n=top_n_words)]
        pc_c  = sum(1 for w in top_words if '_PC_' in w)
        ps_c  = sum(1 for w in top_words if '_PS_' in w)
        bot_c = sum(1 for w in top_words if '_BOTH_' in w)
        like_c = sum(1 for w in top_words if w.endswith('_LIKE'))
        dis_c  = sum(1 for w in top_words if w.endswith('_DISLIKE'))
        dominant = max({'PC':pc_c,'PS':ps_c,'BOTH':bot_c}, key=lambda x:{'PC':pc_c,'PS':ps_c,'BOTH':bot_c}[x])
        genres = [w.split('_')[3] for w in top_words if len(w.split('_'))>=5]
        top_genre = pd.Series(genres).value_counts().index[0] if genres else '?'
        print(f'\n--- Topic {tid} ---')
        print(f'  Dominant platform : {dominant}  (PC={pc_c}, PS={ps_c}, BOTH={bot_c})')
        print(f'  Top genre         : {top_genre}')
        print(f'  Sentiment         : {like_c} LIKE vs {dis_c} DISLIKE')
        print(f'  Top tokens:')
        for w in top_words: print(f'    {w}')

    # Cross-tab
    print('\n' + '='*65)
    print('  True Segment vs Most-Probable LDA Topic')
    print('='*65)
    uid_list = list(user_docs.keys())
    rows = [{'true_segment': int(uid_list[i].split('_')[0]),
             'lda_topic': int(np.argmax(doc.get_topic_dist()))}
            for i, doc in enumerate(lda.docs)]
    ct = pd.crosstab(pd.DataFrame(rows)['true_segment'],
                     pd.DataFrame(rows)['lda_topic'],
                     rownames=['True Segment'], colnames=['LDA Topic'])
    print(ct)
    print('\nDone.')

In [15]:
learn_segments(ratings_file='ratings_simple.csv', games=games, k=5, n_iter=500)

Documents (unique users) : 267
Avg tokens per document  : 37.5

Anchor sizes: PC=20, PS=20, BOTH=20, Budget=20, Casual=20

Training LDA...
  Iteration   50  |  log-likelihood: -6.1897
  Iteration  100  |  log-likelihood: -6.1349
  Iteration  150  |  log-likelihood: -6.1050
  Iteration  200  |  log-likelihood: -6.1098
  Iteration  250  |  log-likelihood: -6.0918
  Iteration  300  |  log-likelihood: -6.0882
  Iteration  350  |  log-likelihood: -6.0676
  Iteration  400  |  log-likelihood: -6.0525
  Iteration  450  |  log-likelihood: -6.0543
  Iteration  500  |  log-likelihood: -6.0468

  LDA DISCOVERED TOPICS

--- Topic 0 ---
  Dominant platform : BOTH  (PC=0, PS=0, BOTH=10)
  Top genre         : Strategy
  Sentiment         : 10 LIKE vs 0 DISLIKE
  Top tokens:
    Game_0065_BOTH_Simulation_LIKE
    Game_0160_BOTH_Horror_LIKE
    Game_0053_BOTH_Strategy_LIKE
    Game_0052_BOTH_Strategy_LIKE
    Game_0194_BOTH_Adventure_LIKE
    Game_0198_BOTH_RPG_LIKE
    Game_0027_BOTH_RPG_LIKE
    Game_

---
## Conclusion — Phase 1

With simple criteria the **platform split is very clean** — Segments 1, 2, and 3
produce tokens with distinct `_PC_`, `_PS_`, `_BOTH_` markers that LDA picks up immediately.

Segments 4 (Budget) and 5 (Casual) are harder because cheap games and family-rated
games are spread across all platforms, so their tokens overlap with the platform segments.

**→ See Phase 2 notebook where harder criteria are added to fix this.**